# Example 11.3.1 / 3: Functions Inside Optimization Expressions

Companion notebook for **11.3.1 Imported Functions and GSL** in the AMPL lab Documentation, section "Functions Inside Optimization Expressions". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Functions Inside Optimization Expressions

When an imported function contains variables, the solver needs function evaluations and may need first or second derivatives. A library function that is useful in a script is therefore not automatically suitable inside an objective or constraint. Confirm continuity, derivative support, domain, and solver compatibility.

The same imported function inside an objective, sent to two solvers. Ipopt evaluates the GSL
function and its derivatives and returns a local minimizer of $\Phi(x)+0.1x^2$; HiGHS cannot
represent an imported call and rejects the model:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
load amplgsl.dll;
function gsl_cdf_ugaussian_P;  # standard normal CDF

var x >= -3, <= 3;  # bounded argument

# needs a solver that evaluates imported functions
minimize Obj: gsl_cdf_ugaussian_P(x) + 0.1 * x^2;

Writing model.mod


### amplpy: run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# run.py
ampl.solve(solver="ipopt", verbose=False)
print("ipopt:", ampl.solve_result, round(ampl.var["x"].value(), 6))
ampl.solve(solver="highs", verbose=False)
print("highs:", ampl.solve_result)
print(ampl.get_value("solve_message"))

ipopt: solved -1.095116
highs: failure
HiGHS 1.14.0: Adding constraint of type 'Call' to HighsModelAPI: Not handling constraint type 'Call'. Provide a handler or a converter method

